In [3]:
pip install opacus

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
from torchvision import datasets
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor
from torch.utils.data import Subset
from torch.utils.data import DataLoader
import torch.nn as nn
from torch import optim
from opacus import PrivacyEngine
from torch.autograd import Variable
import numpy as np

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

In [7]:
train_data = CIFAR10(
    root = 'data',
    train = True,
    transform = transform,
    download = True,
)
train_public = Subset(train_data, list(range(2000)))
train_private = Subset(train_data, list(range(2000, len(train_data))))
test_data = CIFAR10(
    root = 'data',
    train = False,
    transform = transform
)

In [8]:
loaders = {
    'train_public' : torch.utils.data.DataLoader(train_public,
                                          batch_size=100,
                                          shuffle=True,),
    'train_private' : torch.utils.data.DataLoader(train_private,
                                          batch_size=100,
                                          shuffle=True,),
    'test'  : torch.utils.data.DataLoader(test_data,
                                          batch_size=100,
                                          shuffle=True,),
}

In [9]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.out = nn.Linear(64 * 4 * 4, 10)
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        output = self.out(x)
        return output, x

In [10]:
cnn = CNN()

In [11]:
lossf = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr = 0.001)
def acc(predictions,labels):
    return (predictions==labels).mean()

In [12]:
def train(num_epochs, cnn, data_loader):
    cnn.train()
    total_step = len(data_loader)
    for epoch in range(num_epochs):
        overall_acc = []
        overall_loss = []
        for i, (images, labels) in enumerate(data_loader):
            b_x = Variable(images)
            b_y = Variable(labels)
            output = cnn(b_x)[0]
            predictions = np.argmax(output.detach().cpu().numpy(), axis=1)
            labels = labels.detach().cpu().numpy()
            accuracy = acc(predictions,labels)
            overall_acc.append(accuracy)
            loss = lossf(output, b_y)
            overall_loss.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print('Epoch [{}/{}], Step [{}/{}], Accuracy: {:.4f}, Loss: {:.4f}'.format(epoch + 1, num_epochs, i + 1, total_step, np.mean(overall_acc) * 100, np.mean(overall_loss)))

In [13]:
train(15, cnn, loaders['train_public'])

Epoch [1/15], Step [20/20], Accuracy: 21.0000, Loss: 2.1535
Epoch [2/15], Step [20/20], Accuracy: 33.7000, Loss: 1.8540
Epoch [3/15], Step [20/20], Accuracy: 39.7500, Loss: 1.6771
Epoch [4/15], Step [20/20], Accuracy: 44.4500, Loss: 1.5324
Epoch [5/15], Step [20/20], Accuracy: 48.7000, Loss: 1.4326
Epoch [6/15], Step [20/20], Accuracy: 53.0000, Loss: 1.3221
Epoch [7/15], Step [20/20], Accuracy: 56.1000, Loss: 1.2464
Epoch [8/15], Step [20/20], Accuracy: 59.0000, Loss: 1.1723
Epoch [9/15], Step [20/20], Accuracy: 62.7500, Loss: 1.0670
Epoch [10/15], Step [20/20], Accuracy: 65.3500, Loss: 1.0125
Epoch [11/15], Step [20/20], Accuracy: 69.6000, Loss: 0.9197
Epoch [12/15], Step [20/20], Accuracy: 73.8500, Loss: 0.7750
Epoch [13/15], Step [20/20], Accuracy: 76.8000, Loss: 0.7005
Epoch [14/15], Step [20/20], Accuracy: 81.5500, Loss: 0.5794
Epoch [15/15], Step [20/20], Accuracy: 84.4000, Loss: 0.4907


In [14]:
privacy_engine = PrivacyEngine()
cnn, optimizer, loaders['train_private'] = privacy_engine.make_private_with_epsilon(
    module=cnn,
    optimizer=optimizer,
    data_loader=loaders['train_private'],
    epochs=25,
    target_epsilon=8,
    target_delta=1e-5,
    max_grad_norm=1.0,
)

/opt/anaconda3/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


In [15]:
train(25, cnn, loaders['train_private'])

Epoch [1/25], Step [480/480], Accuracy: 49.1864, Loss: 1.8334
Epoch [2/25], Step [480/480], Accuracy: 52.6461, Loss: 1.7518
Epoch [3/25], Step [480/480], Accuracy: 53.5582, Loss: 1.7522
Epoch [4/25], Step [480/480], Accuracy: 54.7505, Loss: 1.7506
Epoch [5/25], Step [480/480], Accuracy: 55.3071, Loss: 1.7878
Epoch [6/25], Step [480/480], Accuracy: 55.6287, Loss: 1.8102
Epoch [7/25], Step [480/480], Accuracy: 56.0109, Loss: 1.8514
Epoch [8/25], Step [480/480], Accuracy: 56.6374, Loss: 1.8607
Epoch [9/25], Step [480/480], Accuracy: 57.3553, Loss: 1.8677
Epoch [10/25], Step [480/480], Accuracy: 57.0150, Loss: 1.9305
Epoch [11/25], Step [480/480], Accuracy: 58.0875, Loss: 1.9230
Epoch [12/25], Step [480/480], Accuracy: 57.7189, Loss: 1.9541
Epoch [13/25], Step [480/480], Accuracy: 58.6839, Loss: 1.9522
Epoch [14/25], Step [480/480], Accuracy: 58.4419, Loss: 2.0084
Epoch [15/25], Step [480/480], Accuracy: 58.2715, Loss: 1.9720
Epoch [16/25], Step [480/480], Accuracy: 59.0914, Loss: 1.9802
E

In [16]:
def test():
    cnn.eval()
    overall_acc = []
    overall_loss = []
    with torch.no_grad():
        for images, labels in loaders['test']:
            test_output, last_layer = cnn(images)
            pred_y = torch.max(test_output, 1)[1].data.squeeze()
            labels = Variable(labels)
            loss = lossf(test_output, labels)
            overall_loss.append(loss.item())
            accuracy = (pred_y == labels).sum().item() / float(labels.size(0))
            labels = labels.detach().cpu().numpy()
            overall_acc.append(accuracy)
            pass
        print('Test Accuracy: %.4f' % (np.mean(overall_acc) * 100) + ', Test Loss: %.4f' % np.mean(overall_loss))
    pass

In [17]:
test()

Test Accuracy: 57.2400, Test Loss: 2.3098
